# OST vs peppr lDDT-PLI (NEOFOLD, fixed)

Comparing per-ligand lDDT-PLI scores between:

- **OST** ([run 56082](https://metaflow.internal.vantai.io/MoleculearnRunsNPosesEvalFlow/56082)): `ost compare-ligand-structures` on NEOFOLD predictions
- **peppr (fixed)**: local evaluation using `runs_n_poses_eval_local.py` with the `compute_lddt_pli_for_mapping` bug fix

### What was fixed

`compute_lddt_pli_for_mapping` (inside `_find_optimal_match_precise` in `match.py`) called
`struc.lddt(reference, pose)` with arrays of different sizes when the model had extra unmatched
ligand chains. This raised `IndexError`, was silently caught, and fell back to RMSD-only polymer
chain selection. For symmetric homodimers, RMSD could pick the **wrong** chain assignment (lower
RMSD ≠ correct mapping), placing monoatomic ions (MG, ZN) 30+ Å from the reference → lDDT-PLI=0.

**Fix**: call `filter_matched(reference, pose)` first to get equal-sized 1:1 corresponding arrays
before computing lDDT-PLI. Committed as `21dc3f7` on branch `fix/bisyrmsd-interface-mask`.

### Data

- OST run 56082: `gs://vantq-argo/moleculearn/infer/eval/MoleculearnRunsNPosesEvalFlow/56082/scores.parquet`
- peppr: `/data/douglas/src/peppr-internal/ost_test/scores.parquet` (1755 rows, 217 systems × 5 models)

In [ ]:
import polars as pl
import plotly.express as px
from pathlib import Path

In [ ]:
# OST per-ligand scores for NEOFOLD predictions (run 56082)
ost_df = (
    pl.read_parquet(
        "gs://vantq-argo/moleculearn/infer/eval/MoleculearnRunsNPosesEvalFlow/56082/scores.parquet"
    )
)
print(ost_df.schema)
print(f"OST rows: {len(ost_df)}")
ost_df.head()

In [ ]:
# New peppr per-ligand scores from local evaluation (fixed lDDT-PLI)
peppr_df = (
    pl.read_parquet("/data/douglas/src/peppr-internal/ost_test/scores.parquet")
    .with_columns(
        pl.col("system_id").str.split("::").list.get(0).alias("system_name"),
        pl.col("system_id").str.split("::").list.get(1).alias("model_id"),
        pl.col("system_id").str.split("::").list.get(2).alias("target_ligand_chain"),
    )
    .rename({"polymer-ligand lDDT": "peppr_lddt_pli", "PLI-LRMSD": "peppr_rmsd"})
    .select("system_name", "model_id", "target_ligand_chain", "peppr_lddt_pli", "peppr_rmsd")
)
print(f"peppr rows: {len(peppr_df)}")
peppr_df.head()

In [ ]:
# Normalize OST columns — run 56082 uses 'model_num' and pivoted metrics
# Detect schema and normalize to: system_name, model_id, target_ligand_chain, ost_lddt_pli, ost_rmsd
if "model_num" in ost_df.columns:
    ost_norm = ost_df.rename({"model_num": "model_id"})
elif "model_id" in ost_df.columns:
    ost_norm = ost_df
else:
    raise ValueError(f"Unexpected OST columns: {ost_df.columns}")

# If metrics are in long format (columns: metrics, value), pivot first
if "metrics" in ost_norm.columns and "value" in ost_norm.columns:
    ost_norm = (
        ost_norm
        .with_columns(
            pl.col("target_ligand_chain").map_elements(lambda x: Path(x).stem, return_dtype=pl.Utf8),
            (pl.lit("ost_") + pl.col("metrics")).alias("metrics"),
        )
        .pivot(index=["model_id", "system_name", "target_ligand_chain"], on="metrics", values="value")
    )
elif "lddt_pli" in ost_norm.columns:
    # Already wide format
    ost_norm = ost_norm.rename({"lddt_pli": "ost_lddt_pli", "rmsd": "ost_rmsd"})

print(f"OST normalized columns: {ost_norm.columns}")
ost_norm.head()

In [ ]:
# Join peppr and OST on system_name + model_id + target_ligand_chain
compare_df = peppr_df.join(
    ost_norm.select("system_name", "model_id", "target_ligand_chain", "ost_lddt_pli", "ost_rmsd"),
    on=["system_name", "model_id", "target_ligand_chain"],
    how="inner",
).drop_nulls(subset=["peppr_lddt_pli", "ost_lddt_pli"])

print(f"Joined rows: {len(compare_df)}")
compare_df.head()

In [ ]:
fig = px.scatter(
    data_frame=compare_df.to_pandas(),
    x="ost_lddt_pli",
    y="peppr_lddt_pli",
    width=600,
    height=600,
    template="plotly_dark",
    title="OST vs peppr lDDT-PLI (NEOFOLD, fixed)",
    trendline="ols",
    hover_data=["model_id", "system_name", "target_ligand_chain"],
    labels={"ost_lddt_pli": "OST lDDT-PLI", "peppr_lddt_pli": "peppr lDDT-PLI"},
)
fig.add_shape(
    type="line",
    x0=0, y0=0, x1=1, y1=1,
    line=dict(color="red", width=2, dash="dash"),
)

# Compute and display Pearson r
import numpy as np
valid = compare_df.select(["ost_lddt_pli", "peppr_lddt_pli"]).to_numpy()
r = np.corrcoef(valid[:, 0], valid[:, 1])[0, 1]
fig.add_annotation(
    x=0.05, y=0.95, xref="paper", yref="paper",
    text=f"Pearson r = {r:.4f}<br>n = {len(valid)}",
    showarrow=False,
    font=dict(size=14),
    align="left",
)

fig.show()